### 評価値の算出

In [ ]:
import glob
import os
import pandas as pd


def process_csv_folder(target_dir: str):
    """指定フォルダ内の全CSVを読み込み、評価値(eval_value_pct)を追加・保存する関数"""
    # 前後の空白や引用符 (' や ") を自動除去
    target_dir = target_dir.strip().strip("'\"")

    if not os.path.exists(target_dir):
        print(f"エラー: 指定されたフォルダ '{target_dir}' が存在しません。")
        return

    csv_files = glob.glob(os.path.join(target_dir, "*.csv"))

    if not csv_files:
        print(
            f"指定されたフォルダ '{target_dir}' 内にCSVファイルが見つかりませんでした。"
        )
        return

    print(f"合計 {len(csv_files)} 件のCSVファイルを処理します...\n")

    for file_path in csv_files:
        df = pd.read_csv(file_path)

        # カラム名の自動識別
        comp_col = next(
            (c for c in ["comp_with_false", "sigma_comp"] if c in df.columns),
            None,
        )
        indep_col = next(
            (
                c
                for c in ["indep_with_false", "sigma_indep"]
                if c in df.columns
            ),
            None,
        )

        if comp_col and indep_col:
            # 評価式: (1 - (indep_with_false / comp_with_false)) * 100
            df["eval_value_pct"] = (1 - (df[indep_col] / df[comp_col])) * 100
            df["eval_value_pct"] = df["eval_value_pct"].round(2)

            df.to_csv(file_path, index=False)
            print(f"【更新成功】 {os.path.basename(file_path)}")
        else:
            print(
                f"【スキップ】 {os.path.basename(file_path)} : 対象の列名が存在しません。"
            )

    print("\nすべてのファイルの処理が完了しました。")

In [ ]:
target_folder = input(
    "../../data/LFR01/result/f_com1_user_2"
)
process_csv_folder(target_folder)